### Load Model

In [1]:
import torch
from neurojepa.utils.init_utils import load_backbone_from_hf

device = "cuda" if torch.cuda.is_available() else "cpu"

backbone = load_backbone_from_hf(
    "NYUMedML/Neuro-JEPA",
    device=device,
)

backbone.eval()

[INFO    ][2026-06-04 22:29:46][httpx               ][_send_single_request     ] HTTP Request: HEAD https://huggingface.co/NYUMedML/Neuro-JEPA/resolve/main/config.json "HTTP/1.1 200 OK"
[INFO    ][2026-06-04 22:29:46][root                ][_load_hf_or_local_config ] Loading Neuro-JEPA backbone config from /gpfs/home/huangh13/.cache/huggingface/hub/models--NYUMedML--Neuro-JEPA/snapshots/0200d0d2a0457f41113fe43e4adae18131f3d816/config.json
MOE layer indices: [1, 3, 5, 7, 9, 11]
[INFO    ][2026-06-04 22:29:47][root                ][init_backbone            ] VisionTransformer(
  (patch_embed): PatchEmbed3D(
    (proj): Conv3d(1, 768, kernel_size=(12, 12, 12), stride=(12, 12, 12))
  )
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): RoPEAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features

VisionTransformer(
  (patch_embed): PatchEmbed3D(
    (proj): Conv3d(1, 768, kernel_size=(12, 12, 12), stride=(12, 12, 12))
  )
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): RoPEAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
        (proj_attn_gate): Linear(in_features=768, out_features=12, bias=True)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((768,), eps=

### Cretea Dataloader

In [2]:
import numpy as np

from monai import data
from monai import transforms
from monai.transforms import Lambdad

<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [3]:
# ---- Load and preprocess the volume (same pipeline as training) ----
def remove_nan(img):
    img[np.isnan(img)] = 0.0
    return img

roi_size = [96, 108, 96]

trans = transforms.Compose([
    transforms.LoadImaged(keys=['image'], image_only=False),
    transforms.EnsureChannelFirstd(keys=['image']),
    Lambdad(('image',), remove_nan),
    transforms.Orientationd(keys=['image'], axcodes='RAS'),
    transforms.Spacingd(keys=['image'], pixdim=(1.0, 1.0, 1.0), mode=[5]),
    transforms.CropForegroundd(
        keys=['image'], source_key='image',
        select_fn=lambda x: x > 0.0, margin=4, allow_smaller=True),
    transforms.ResizeWithPadOrCropd(
        keys=['image'], spatial_size=[180, 216, 180], mode='edge'),
    transforms.Resized(keys=['image'], spatial_size=[100, 120, 100]),
    transforms.CenterSpatialCropd(
        keys=["image"],
        roi_size=roi_size,
        allow_missing_keys=True,
    ),
    transforms.ScaleIntensityRangePercentilesd(
        keys=['image'], lower=0.5, upper=99.5, b_min=0, b_max=1, clip=True),
    transforms.CastToTyped(
        keys=["image"],
        dtype=np.float32,
        allow_missing_keys=True,
    ),
])

/gpfs/data/denizlab/Users/hh2740/miniconda3/envs/neurojepa/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [4]:
sample_images = [
    {'image': '/path/to/filename1.nii.gz'},
    {'image': '/path/to/filename2.nii.gz'},
]

In [5]:
# Define dataloader and get scan array
batch_size = 2

test_ds = data.Dataset(
    data=sample_images, 
    transform=trans,
)

test_loader = data.DataLoader(
    dataset=test_ds,
    batch_size=batch_size,
    num_workers=1,
    pin_memory=True,
    shuffle=False,
)

x_img = next(iter(test_loader))['image']
x_img = x_img.to(device)

In [6]:
print(f"Shape for current scan: {x_img.shape}")

Shape for current scan: torch.Size([2, 1, 96, 108, 96])


### Extract Feature

In [7]:
torch.backends.cudnn.enabled = False
# Model output last layer and MoE scores per layer by default
with torch.autocast("cuda"):
    outs_features, outs_moe_scores = backbone(x_img)

In [8]:
print(f"Shape of last layer output feature: {outs_features.shape}")

Shape of last layer output feature: torch.Size([2, 576, 768])
